In [4]:
import os
import pandas as pd
import numpy as np

In [5]:
os.chdir(".")

In [6]:
model_results_dir = "AllResults_05142025\AllResults" #"ModelRunResults"

<>:1: SyntaxWarning: invalid escape sequence '\A'
<>:1: SyntaxWarning: invalid escape sequence '\A'
/var/folders/qp/sq840hxs1rs_wtmckrby_mkw0000gn/T/ipykernel_46423/445637031.py:1: SyntaxWarning: invalid escape sequence '\A'
  model_results_dir = "AllResults_05142025\AllResults" #"ModelRunResults"


In [7]:
# Define the data as a list of dictionaries
data = [
    {"pollutantID": 2, "pollutantName": "carbon monoxide (CO)", "PM_Name": "Carbon monoxide (CO) emissions"},
    {"pollutantID": 3, "pollutantName": "oxides of nitrogen (NOx)", "PM_Name": "Nitrous oxide (NOx) emissions"},
    {"pollutantID": 87, "pollutantName": "volatile organic compounds (VOC)", "PM_Name": "Volatile organic compounds (VOC) emissions"},
    {"pollutantID": 98, "pollutantName": "carbon dioxide equivalent (CO2e)", "PM_Name" : "Greenhouse gas emissions (CO2) equivalent"},
    {"pollutantID": 100, "pollutantName": "particulate matter 10 (PM 10) exhaust", "PM_Name": "Particulate matter 10 micrometers or smaller emissions (PM10) "},
    {"pollutantID": 106, "pollutantName": "particulate matter 10 (PM 10) brakewear", "PM_Name": "Particulate matter 10 micrometers or smaller emissions (PM10) "},
    {"pollutantID": 107, "pollutantName": "particulate matter 10 (PM 10) tirewear", "PM_Name": "Particulate matter 10 micrometers or smaller emissions (PM10) "},
    {"pollutantID": 110, "pollutantName": "particulate matter 2.5 (PM 2.5) exhaust", "PM_Name": "Particulate matter 2.5 micrometers or smaller emissions (PM2.5)"},
    {"pollutantID": 116, "pollutantName": "particulate matter 2.5 (PM 2.5) brakewear", "PM_Name": "Particulate matter 2.5 micrometers or smaller emissions (PM2.5)"},
    {"pollutantID": 117, "pollutantName": "particulate matter 2.5 (PM 2.5) tirewear", "PM_Name": "Particulate matter 2.5 micrometers or smaller emissions (PM2.5)"},
]

# Convert the data into a DataFrame
pollutants_df = pd.DataFrame(data)

In [8]:
# Create sample output data for testing
# all_exp = pd.read_csv(os.path.join(r"EMAT\PAG", "pag_emat_design_experiments_v2.csv"))

# all_perf_measure = perf_measures_names + pollutants_perf_measures
# Add each performance measure as a new column to the all experiment data
# for measure in all_perf_measure:
#    all_exp[measure] = np.random.rand()  # Assign random data for testing

# Save the updated DataFrame to a new CSV file
#all_exp.to_csv(os.path.join(r"EMAT\PAG", "pag_emat_design_experiments_v2_with_perf_measures.csv"), index=False)
#print(all_exp.shape)

In [9]:
# following Files have the Scenario name incorrectly coded
file1 = pd.read_csv(os.path.join(model_results_dir, "AllPerformanceMeasures_Run061.csv"))
file1['Scenario'] = 'Run061'
file1.to_csv(os.path.join(model_results_dir, "AllPerformanceMeasures_Run061.csv"), index=False)

file2 = pd.read_csv(os.path.join(model_results_dir, "AllPerformanceMeasures_Run089.csv"))
file2['Scenario'] = 'Run089'
file2.to_csv(os.path.join(model_results_dir, "AllPerformanceMeasures_Run089.csv"), index=False)

#file2 = pd.read_csv(os.path.join(model_results_dir, "AllPerformanceMeasures_Run081.csv"))
#file2['Scenario'] = 'Run081'
#file2.to_csv(os.path.join(model_results_dir, "AllPerformanceMeasures_Run081.csv"), index=False)


FileNotFoundError: [Errno 2] No such file or directory: 'AllResults_05142025\\AllResults/AllPerformanceMeasures_Run061.csv'

In [ ]:
all_perf_files = [f for f in os.listdir(model_results_dir) if "AllPerformanceMeasures" in f]
merged_df = None
all_comb_files = []

for file in all_perf_files:
    run_exp = pd.read_csv(
        os.path.join(model_results_dir, file)
    )
    # Replace NaN with an empty string for the relevant columns
    columns_to_concat = ['source', 'measures', 'dim1', 'dim1_value', 'dim2', 'dim2_value', 'Scenario', 'measure_name']
    run_exp[columns_to_concat] = run_exp[columns_to_concat].fillna('')

    # Create the combined column
    run_exp['comb_col_name'] = run_exp['source'] + "_" + run_exp["measures"] + "_" + run_exp["dim1"] + "_" + run_exp["dim1_value"] + "_" + run_exp['dim2'] + "_" + run_exp['dim2_value'] + \
                                    "_" + run_exp['measure_name']

    # get experiment id
    exp_id  = run_exp["Scenario"].unique()[0]

    # extract the number from the filename
    exp_id_in_filename = file.split("_")[1].split(".")[0]

    # check if exp id is same in filename and within the file
    if exp_id != exp_id_in_filename:
        raise ValueError(f"Experiment ID in filename {exp_id_in_filename} does not match the one in the file {exp_id}.")

    # filter colums
    run_exp = run_exp[["comb_col_name", "measure_value"]]

    # rename columns
    run_exp.columns = ["comb_col_name", exp_id_in_filename]

    # lets merge the current dataframes in each iteration

    if merged_df is not None:
        merged_df = pd.merge(merged_df, run_exp, on='comb_col_name', how='outer')
    else:
        merged_df = run_exp
    
    merged_df.to_csv(os.path.join(model_results_dir, 'merged_results.csv'), index=False)

In [ ]:
# read all files in the directory with 2055_moves4_ in file name
all_moves_files = [f for f in os.listdir(model_results_dir) if "2055_moves4_" in f]

all_moves_df = None

for file in all_moves_files:
    #print(file)
    run_exp_moves = pd.read_csv(
    os.path.join(model_results_dir, file),
    )

    run_exp_moves = run_exp_moves.groupby(['pollutantID'])['emissionQuant'].sum().reset_index()
    run_exp_moves = pd.merge(run_exp_moves, pollutants_df, on='pollutantID', how='left')

    run_exp_moves = run_exp_moves.groupby(['PM_Name'])['emissionQuant'].sum().reset_index()

    # Extract the run ID (e.g., "Run001") from the file name
    run_id = file.split("_")[2].split(".")[0]
    
    #print(run_id)
    run_exp_moves.columns = ['PM_Name', run_id]
    
    if all_moves_df is not None:
        all_moves_df = pd.merge(all_moves_df, run_exp_moves, on='PM_Name', how='outer')
    else:   
        all_moves_df = run_exp_moves
    
    all_moves_df.to_csv(os.path.join(model_results_dir, 'all_moves_results.csv'), index=False)


In [ ]:
all_moves_df

,PM_Name,Run002,Run003,Run004,Run005,Run006,Run007,Run008,Run009,Run010,...,Run111,Run112,Run113,Run114,Run115,Run116,Run117,Run118,Run119,Run120
0,Carbon monoxide (CO) emissions,8.867268e+09,8.786269e+09,9.942429e+09,8.279030e+09,7.164095e+09,7.373905e+09,7.125328e+09,9.768301e+09,8.132079e+09,...,8.932774e+09,9.668737e+09,7.639012e+09,1.005096e+10,7.956115e+09,7.670045e+09,7.193162e+09,6.905879e+09,6.935131e+09,8.354024e+09
1,Greenhouse gas emissions (CO2) equivalent,2.677424e+12,2.568047e+12,2.821317e+12,2.373698e+12,2.050616e+12,2.321689e+12,2.112150e+12,2.756208e+12,2.356509e+12,...,2.594043e+12,2.892610e+12,2.195695e+12,2.890148e+12,2.385862e+12,2.259786e+12,2.114924e+12,2.050826e+12,2.053526e+12,2.370755e+12
2,Nitrous oxide (NOx) emissions,7.789986e+08,7.128425e+08,7.294010e+08,6.718648e+08,6.172412e+08,7.413647e+08,6.627594e+08,7.132483e+08,6.804412e+08,...,7.209285e+08,7.695368e+08,6.349273e+08,7.315063e+08,6.937762e+08,6.784860e+08,6.452707e+08,6.408822e+08,6.505723e+08,6.513797e+08
3,Particulate matter 10 micrometers or smaller e...,4.123554e+08,3.513231e+08,3.287461e+08,3.056151e+08,2.791315e+08,3.987978e+08,3.082744e+08,3.294265e+08,3.068956e+08,...,3.269245e+08,3.905741e+08,2.936580e+08,3.495327e+08,3.424663e+08,3.235025e+08,3.131378e+08,3.140870e+08,3.094988e+08,2.945603e+08
4,Particulate matter 2.5 micrometers or smaller ...,7.637441e+07,6.941114e+07,6.875149e+07,6.232194e+07,5.689198e+07,7.170773e+07,5.964075e+07,6.905333e+07,6.186726e+07,...,6.585462e+07,7.607713e+07,6.007857e+07,7.263552e+07,6.650744e+07,6.305976e+07,6.116640e+07,6.056612e+07,5.956499e+07,6.195391e+07
5,Volatile organic compounds (VOC) emissions,9.965334e+08,1.031391e+09,1.112212e+09,9.738157e+08,9.034574e+08,8.966381e+08,8.642368e+08,1.136224e+09,9.471081e+08,...,9.961184e+08,1.096598e+09,9.564173e+08,1.191368e+09,9.635166e+08,9.198248e+08,9.055693e+08,8.806014e+08,8.598993e+08,1.027662e+09


In [ ]:
perf_df = pd.read_excel(
    os.path.join("crosswalks", "perf_measures_names.xlsx")
)
perf_measures_names = dict(zip(perf_df['comb_col_name'], perf_df['PF Name in the Dashboard']))

merged_df['perf_measures'] = merged_df['comb_col_name'].map(perf_measures_names)

#drop the comb_col_name column
merged_df = merged_df.drop(columns=['comb_col_name'])

C:\Users\vyadav\AppData\Local\Temp\ipykernel_6460\264178040.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_df['perf_measures'] = merged_df['comb_col_name'].map(perf_measures_names)


In [ ]:
# Transpose merged_df
transposed_merged_df = merged_df.set_index('perf_measures').transpose()
transposed_merged_df.index.name = 'RunID'

# Transpose all_moves_df
transposed_all_moves_df = all_moves_df.set_index('PM_Name').transpose()
transposed_all_moves_df.index.name = 'RunID'

# Merge the two dataframes based on RunID
merged_transposed_df = transposed_merged_df.merge(transposed_all_moves_df, left_index=True, right_index=True, how='outer')

# Display the merged dataframe
merged_transposed_df

,Regional Auto Trips,Regional Bike Trips,Regional School Bus Trips,Regional Taxi Trips,Regional Transit Trips,Regional Walk Trips,Auto Mode Share,Bike Mode Share,School Bus Mode Share,Taxi Mode Share,...,Disadvantage Zone Transit Accessibility Off Peak within 90 Minutes for Basic Needs 1,Disadvantage Zone Transit Accessibility Peak within 90 Minutes for Basic Needs 0,Disadvantage Zone Transit Accessibility Peak within 90 Minutes for Basic Needs 1,Average Commute Time by School Bus,Carbon monoxide (CO) emissions,Greenhouse gas emissions (CO2) equivalent,Nitrous oxide (NOx) emissions,Particulate matter 10 micrometers or smaller emissions (PM10),Particulate matter 2.5 micrometers or smaller emissions (PM2.5),Volatile organic compounds (VOC) emissions
RunID,,,,,,,,,,,,,,,,,,,,,
Run001,975355.0,20047.0,15311.0,1604.0,15457.0,132933.0,0.840311,0.017271,0.013191,0.001382,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Run002,1077628.0,20596.0,16466.0,2186.0,21529.0,150204.0,0.836272,0.015983,0.012778,0.001696,...,2.070000,3.787692,0.996923,NaN,8.867268e+09,2.677424e+12,778998600.0,412355381.3,76374409.75,9.965334e+08
Run003,1212005.0,23874.0,20593.0,4198.0,21893.0,170243.0,0.834251,0.016433,0.014175,0.002890,...,5.452857,5.530909,1.818182,NaN,8.786269e+09,2.568047e+12,712842510.0,351323111.8,69411143.69,1.031391e+09
Run004,1074101.0,23755.0,17014.0,4203.0,23422.0,155607.0,0.827440,0.018300,0.013107,0.003238,...,4.564000,4.983684,1.631579,NaN,9.942429e+09,2.821317e+12,729401050.0,328746121.1,68751490.85,1.112212e+09
Run005,1066198.0,23406.0,15296.0,3094.0,32171.0,153768.0,0.823998,0.018089,0.011821,0.002391,...,3.875000,3.588182,1.194545,NaN,8.279030e+09,2.373698e+12,671864850.0,305615061.0,62321938.59,9.738157e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Run116,1122042.0,22511.0,16358.0,2933.0,18884.0,159917.0,0.835695,0.016766,0.012183,0.002184,...,3.964286,4.110000,1.411250,NaN,7.670045e+09,2.259786e+12,678486030.0,323502545.4,63059759.86,9.198248e+08
Run117,1037647.0,21130.0,16824.0,1721.0,19027.0,144958.0,0.835931,0.017022,0.013553,0.001386,...,2.722500,2.665000,1.010000,NaN,7.193162e+09,2.114924e+12,645270690.0,313137774.9,61166397.87,9.055693e+08
Run118,1020203.0,19628.0,16752.0,1935.0,19032.0,137345.0,0.839746,0.016156,0.013789,0.001593,...,3.216000,4.255000,1.350000,NaN,6.905879e+09,2.050826e+12,640882200.0,314086965.5,60566120.59,8.806014e+08


In [ ]:
#save the merged dataframe to a CSV file
merged_transposed_df.to_csv(os.path.join(model_results_dir, 'final_merged_results.csv'), index=True)